## Logistic Regression Baseline

We start with Logistic Regression as a simple and interpretable baseline for predicting SECOM labels.  
Since the main goal is to detect the `1` class correctly, we focus on recall for `1` rather than overall accuracy.  
A high recall for `1` means fewer false negatives, which is important when missing a defective sample is more costly than flagging an extra one.  
We also use `class_weight='balanced'` to handle the class imbalance in the dataset.

In [7]:
import os
import joblib
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, confusion_matrix, classification_report

save_dir = "prepared_pca_datasets"

pca_datasets = {}
for thr in [0.80, 0.90, 0.95]:
    thr_name = int(thr * 100)

    X_train_pca = joblib.load(os.path.join(save_dir, f"X_train_pca_{thr_name}.pkl"))
    X_test_pca  = joblib.load(os.path.join(save_dir, f"X_test_pca_{thr_name}.pkl"))
    y_train     = joblib.load(os.path.join(save_dir, f"y_train_{thr_name}.pkl"))
    y_test      = joblib.load(os.path.join(save_dir, f"y_test_{thr_name}.pkl"))

    pca_datasets[thr] = {
        "X_train": X_train_pca,
        "X_test": X_test_pca,
        "y_train": y_train,
        "y_test": y_test
    }

logreg_results = {}

for thr, data in pca_datasets.items():
    X_train_pca = data["X_train"]
    X_test_pca = data["X_test"]
    y_train_local = data["y_train"]
    y_test_local = data["y_test"]

    model = LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        solver="liblinear",
        random_state=42
    )

    model.fit(X_train_pca, y_train_local)

    y_prob = model.predict_proba(X_test_pca)[:, 1]
    y_pred = np.where(y_prob >= 0.3, 1, -1)

    recall_pos1 = recall_score(y_test_local, y_pred, pos_label=1)

    logreg_results[thr] = {
        "model": model,
        "recall_pos1": recall_pos1
    }

    print(f"\n=== Logistic Regression | PCA threshold: {thr:.2f} ===")
    print(f"Recall for label +1 at threshold 0.3: {recall_pos1:.4f}")
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test_local, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test_local, y_pred))


=== Logistic Regression | PCA threshold: 0.80 ===
Recall for label +1 at threshold 0.3: 0.5238

Confusion Matrix:
[[175 118]
 [ 10  11]]

Classification Report:
              precision    recall  f1-score   support

          -1       0.95      0.60      0.73       293
           1       0.09      0.52      0.15        21

    accuracy                           0.59       314
   macro avg       0.52      0.56      0.44       314
weighted avg       0.89      0.59      0.69       314


=== Logistic Regression | PCA threshold: 0.90 ===
Recall for label +1 at threshold 0.3: 0.3333

Confusion Matrix:
[[215  78]
 [ 14   7]]

Classification Report:
              precision    recall  f1-score   support

          -1       0.94      0.73      0.82       293
           1       0.08      0.33      0.13        21

    accuracy                           0.71       314
   macro avg       0.51      0.53      0.48       314
weighted avg       0.88      0.71      0.78       314


=== Logistic Regressi

In [9]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, confusion_matrix, classification_report

rf_results = {}

for thr, data in pca_datasets.items():
    X_train_pca = data["X_train"]
    X_test_pca = data["X_test"]
    y_train_local = data["y_train"]
    y_test_local = data["y_test"]

    model = RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    )

    model.fit(X_train_pca, y_train_local)

    y_prob = model.predict_proba(X_test_pca)[:, 1]
    y_pred = np.where(y_prob >= 0.3, 1, -1)

    recall_pos1 = recall_score(y_test_local, y_pred, pos_label=1)

    rf_results[thr] = {
        "model": model,
        "recall_pos1": recall_pos1
    }

    print(f"\n=== Random Forest | PCA threshold: {thr:.2f} ===")
    print(f"Recall for label +1 at threshold 0.3: {recall_pos1:.4f}")
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test_local, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test_local, y_pred, zero_division=0))


=== Random Forest | PCA threshold: 0.80 ===
Recall for label +1 at threshold 0.3: 0.0000

Confusion Matrix:
[[293   0]
 [ 21   0]]

Classification Report:
              precision    recall  f1-score   support

          -1       0.93      1.00      0.97       293
           1       0.00      0.00      0.00        21

    accuracy                           0.93       314
   macro avg       0.47      0.50      0.48       314
weighted avg       0.87      0.93      0.90       314


=== Random Forest | PCA threshold: 0.90 ===
Recall for label +1 at threshold 0.3: 0.0000

Confusion Matrix:
[[293   0]
 [ 21   0]]

Classification Report:
              precision    recall  f1-score   support

          -1       0.93      1.00      0.97       293
           1       0.00      0.00      0.00        21

    accuracy                           0.93       314
   macro avg       0.47      0.50      0.48       314
weighted avg       0.87      0.93      0.90       314


=== Random Forest | PCA threshold

In [10]:
print("Max predicted probability for class 1:", y_prob.max())
print("Distribution:", np.percentile(y_prob, [50, 75, 90, 95, 99, 100]))

Max predicted probability for class 1: 0.20666666666666667
Distribution: [0.06       0.07666667 0.09666667 0.10783333 0.13666667 0.20666667]
